In [26]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [35]:
!pip install tensorboard



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 5.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 12.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [tensorboard] [tensorboard]


In [27]:
import torch

if torch.cuda.is_available():
    
    # Если мы в Colab или на сервере с NVIDIA
    device = torch.device('cuda')
    print("Ура, найдена CUDA!")
elif torch.backends.mps.is_available():
    # Если мы на MacBook с чипом (M1/M2/M3...)
    device = torch.device('mps')
    print("Ура, найден чип Apple MPS!")
else:
    # Запасной вариант (обычный процессор)
    device = torch.device('cpu')
    print("Работаем на обычном процессоре (CPU).")


Ура, найден чип Apple MPS!


In [28]:
import sys
import os
# Указываем абсолютный путь к папке seminar/neuralrec (где лежит папка neuralrec)
project_path = "/Users/amirhamidullin/PycharmProjects/recsys/DeepRecSys/week_04/seminar/neuralrec"
if project_path not in sys.path:
    sys.path.append(project_path)
    

In [29]:
from dataclasses import dataclass


@dataclass
class PipelineConfig:
    batch_size: int = 4
    d_model: int = 64
    n_heads: int = 4
    dropout: float = 0.1
    dim_feedforward: int = 256
    num_layers: int = 2
    num_epochs: int = 2
    gradient_clipping: float = 1.0
    logging_every_n_steps: int = 10
    validation_every_n_steps: int = 20
    tensorboard_every_n_steps: int = 10
    learning_rate: float = 1e-3


config = PipelineConfig()

# Yambda

In [38]:
import torch

# Определяем устройство прямо здесь, чтобы не зависеть от порядка ячеек
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

from neuralrec.data.dataloader import DataLoader
from neuralrec.data.transforms import ToDevice
from neuralrec.ext.yambda.utils import load_yambda_user_split
from neuralrec.ext.yambda.transforms import pad_collate_item_id


train_dataset, valid_dataset, max_item_id = load_yambda_user_split()

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=pad_collate_item_id,
    transforms=ToDevice(device)
)
valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=pad_collate_item_id,
    transforms=ToDevice(device),
)


Using device: mps


# Model 

In [39]:
import torch
import torch.nn as nn

from neuralrec.nn.transformer import TransformerEncoder


class RecommenderTransformer(nn.Module):
    def __init__(
        self,
        num_items: int,
        d_model: int,
        encoder: nn.Module,
    ):
        super().__init__()
        self.num_items = num_items
        self.embedding = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.encoder = encoder
        self.to_logits = nn.Linear(d_model, num_items + 1)

    def forward(self, batch) -> dict[str, torch.Tensor]:
        item_id = batch['item_id']
        x = self.embedding(item_id)
        h = self.encoder(x)
        logits = self.to_logits(h)

        logits_pred = logits[:, :-1].contiguous().view(-1, self.num_items + 1)
        target = item_id[:, 1:].contiguous().view(-1)

        loss = nn.functional.cross_entropy(
            logits_pred, target, ignore_index=0, reduction="mean"
        )
        return {"loss": loss}

# Train Loop

In [40]:
from neuralrec.run.train import TrainRunner
from neuralrec.run.callbacks import (
    GradientNormClippingCallback,
    LoggingCallback,
    ValidationCallback,
    TensorBoardCallback,
)

model = RecommenderTransformer(
    num_items=max_item_id,
    d_model=config.d_model,
    encoder=TransformerEncoder(
        d_model=config.d_model,
        nhead=config.n_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
        causal=True,
    )
).to(device)



optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

TrainRunner(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    callbacks=[
        GradientNormClippingCallback(config.gradient_clipping),
        ValidationCallback(valid_loader).every_n_steps(config.validation_every_n_steps),
        LoggingCallback().every_n_steps(config.logging_every_n_steps),
        TensorBoardCallback().every_n_steps(config.tensorboard_every_n_steps),
    ],
).fit(num_epochs=config.num_epochs)

2026-04-16 19:28:19.476 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.9479
2026-04-16 19:28:25.976 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.7990 validation/loss=13.791272048950196
2026-04-16 19:28:29.892 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.7139
2026-04-16 19:28:36.347 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=13.2941 validation/loss=13.302471275329589
2026-04-16 19:28:39.585 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=50 epoch=0 loss=12.5603
2026-04-16 19:28:44.765 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=60 epoch=0 loss=12.3693 validation/loss=12.636831741333008
2026-04-16 19:28:47.703 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=70 epoch=0 loss=12.7269
2026-04-16 19:28:52.873 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step

KeyboardInterrupt: 

# Mixed Precision

In [ ]:
from neuralrec.nn.autocast import AutoCast

model = RecommenderTransformer(
    num_items=max_item_id,
    d_model=config.d_model,
    encoder=TransformerEncoder(
        d_model=config.d_model,
        nhead=config.n_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
        causal=True,
    )
).cuda()
model = AutoCast(model)  # if using fp16, use GradScaler

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, fused=True)

TrainRunner(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    callbacks=[
        GradientNormClippingCallback(config.gradient_clipping),
        ValidationCallback(valid_loader).every_n_steps(config.validation_every_n_steps),
        LoggingCallback().every_n_steps(config.logging_every_n_steps),
    ],
).fit(num_epochs=config.num_epochs)

/home/matveevar/seminar/neuralrec/neuralrec/nn/transformer.py:33: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(
2026-02-24 12:23:40.783 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.9110
2026-02-24 12:23:42.451 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.7927 validation/loss=13.7982368850708
2026-02-24 12:23:43.300 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.6811
2026-02-24 12:23:44.922 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=13.3451 validation/loss=13.322362365722656
2026-02-24 12:23:45.780 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=50 epoch=0 loss=12.6244
2026-02-24 12:23:47.417 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=60 epoch=0 loss=12.2915 val

# Distributed Data Parallel

In [ ]:
%%writefile train_ddp.py
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

from neuralrec.nn.transformer import TransformerEncoder
from neuralrec.data.dataloader import DataLoader
from neuralrec.data.transforms import ToDevice
from neuralrec.ext.yambda.utils import load_yambda_user_split
from neuralrec.ext.yambda.transforms import pad_collate_item_id
from neuralrec.run.train import TrainRunner
from neuralrec.run.callbacks import (
    GradientNormClippingCallback,
    LoggingCallback,
    ValidationCallback,
    TensorBoardCallback,
)
from neuralrec.nn.autocast import AutoCast
from neuralrec.run.distributed import (
    init_process_group,
    destroy_process_group,
    is_chief,
)


@dataclass
class PipelineConfig:
    batch_size: int = 1
    d_model: int = 64
    n_heads: int = 4
    dim_feedforward: int = 256
    dropout: float = 0.1
    num_layers: int = 2
    num_epochs: int = 1
    gradient_clipping: float = 1.0
    logging_every_n_steps: int = 10
    validation_every_n_steps: int = 20
    tensorboard_every_n_steps: int = 10
    learning_rate: float = 1e-3


class RecommenderTransformer(nn.Module):
    def __init__(
        self,
        num_items: int,
        d_model: int,
        encoder: nn.Module,
    ):
        super().__init__()
        self.num_items = num_items
        self.embedding = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.encoder = encoder
        self.to_logits = nn.Linear(d_model, num_items + 1)

    def forward(self, batch) -> dict[str, torch.Tensor]:
        item_id = batch['item_id']
        x = self.embedding(item_id)
        h = self.encoder(x)
        logits = self.to_logits(h)

        logits_pred = logits[:, :-1].contiguous().view(-1, self.num_items + 1)
        target = item_id[:, 1:].contiguous().view(-1)

        loss = nn.functional.cross_entropy(
            logits_pred, target, ignore_index=0, reduction="mean"
        )
        return {"loss": loss}


def main():
    init_process_group()

    config = PipelineConfig()
    train_dataset, valid_dataset, max_item_id = load_yambda_user_split()

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=pad_collate_item_id,
        sampler=DistributedSampler(train_dataset),
        transforms=ToDevice(torch.device("cuda"))
    )
    valid_loader = DataLoader(
        dataset=valid_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=pad_collate_item_id,
        sampler=DistributedSampler(valid_dataset),
        transforms=ToDevice(torch.device("cuda")),
    )

    model = RecommenderTransformer(
        num_items=max_item_id,
        d_model=config.d_model,
        encoder=TransformerEncoder(
            d_model=config.d_model,
            nhead=config.n_heads,
            num_layers=config.num_layers,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            causal=True,
        )
    ).cuda()
    model = AutoCast(model)
    model = DDP(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, fused=True)

    TrainRunner(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        callbacks=[
            GradientNormClippingCallback(config.gradient_clipping),
            ValidationCallback(valid_loader).every_n_steps(config.validation_every_n_steps).ignore_if(not is_chief()),
            LoggingCallback().every_n_steps(config.logging_every_n_steps).ignore_if(not is_chief()),
        ],
    ).fit(config.num_epochs)

    destroy_process_group()

if __name__ == "__main__":
    main()

Overwriting train_ddp.py


In [ ]:
!OMP_NUM_THREADS=1 CUDA_VISIBLE_DEVICES="6,7" torchrun --nproc_per_node 2 train_ddp.py

/home/matveevar/seminar/neuralrec/neuralrec/nn/transformer.py:33: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(
/home/matveevar/seminar/neuralrec/neuralrec/nn/transformer.py:33: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(
2026-02-24 12:25:16.537 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.7815
2026-02-24 12:25:21.896 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.7387 validation/loss=13.821580657958984
2026-02-24 12:25:25.832 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.6220
2026-02-24 12:25:30.368 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=13.7461 validation/loss=13.52961734

# FlashAttention


In [ ]:
from neuralrec.nn.flashattn_transformer import TransformerEncoder as FlashTransformerEncoder


model = RecommenderTransformer(
    num_items=max_item_id,
    d_model=config.d_model,
    encoder=FlashTransformerEncoder(
        d_model=config.d_model,
        nhead=config.n_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
        causal=True,
    )
).cuda()
model = AutoCast(model)  # if using fp16, use GradScaler

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, fused=True)

TrainRunner(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    callbacks=[
        GradientNormClippingCallback(config.gradient_clipping),
        ValidationCallback(valid_loader).every_n_steps(config.validation_every_n_steps),
        LoggingCallback().every_n_steps(config.logging_every_n_steps),
    ],
).fit(num_epochs=config.num_epochs)

2026-02-24 12:27:53.866 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.7397
2026-02-24 12:27:55.485 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.7815 validation/loss=13.785020523071289
2026-02-24 12:27:56.334 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.6933
2026-02-24 12:27:57.941 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=13.3355 validation/loss=13.291863975524903
2026-02-24 12:27:58.793 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=50 epoch=0 loss=12.4394
2026-02-24 12:28:00.455 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=60 epoch=0 loss=12.2967 validation/loss=12.639409141540527
2026-02-24 12:28:01.307 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=70 epoch=0 loss=12.5977
2026-02-24 12:28:02.916 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step

# Torch Compile

In [ ]:
model = RecommenderTransformer(
    num_items=max_item_id,
    d_model=config.d_model,
    encoder=FlashTransformerEncoder(
        d_model=config.d_model,
        nhead=config.n_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
        causal=True,
    )
).cuda()
model = AutoCast(model)  # if using fp16, use GradScaler
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, fused=True)

TrainRunner(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    callbacks=[
        GradientNormClippingCallback(config.gradient_clipping),
        ValidationCallback(valid_loader).every_n_steps(config.validation_every_n_steps),
        LoggingCallback().every_n_steps(config.logging_every_n_steps),
    ],
).fit(num_epochs=config.num_epochs)

2026-02-24 12:28:46.597 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.7600
2026-02-24 12:28:47.951 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.7926 validation/loss=13.778024978637696
2026-02-24 12:28:50.993 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.6995
2026-02-24 12:28:51.851 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=13.1980 validation/loss=13.247539176940919
2026-02-24 12:28:52.430 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=50 epoch=0 loss=12.5191
2026-02-24 12:28:53.284 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=60 epoch=0 loss=12.2631 validation/loss=12.620790328979492
2026-02-24 12:28:53.872 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=70 epoch=0 loss=12.6428
2026-02-24 12:28:54.735 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step

# Reader

In [ ]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=8,  # increase number of workers
    pin_memory=True,  # transfer tensors in pinned memory
    collate_fn=pad_collate_item_id,
    transforms=ToDevice(torch.device("cuda"))
)

model = RecommenderTransformer(
    num_items=max_item_id,
    d_model=config.d_model,
    encoder=FlashTransformerEncoder(
        d_model=config.d_model,
        nhead=config.n_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
        causal=True,
    )
).cuda()
model = AutoCast(model)  # if using fp16, use GradScaler

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, fused=True)

TrainRunner(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    callbacks=[
        GradientNormClippingCallback(config.gradient_clipping),
        LoggingCallback().every_n_steps(config.logging_every_n_steps),
    ],
).fit(num_epochs=config.num_epochs)

2026-02-24 12:29:17.420 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.8897
2026-02-24 12:29:18.285 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.7382
2026-02-24 12:29:19.139 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.6595
2026-02-24 12:29:19.986 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=13.1545
2026-02-24 12:29:20.840 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=50 epoch=0 loss=12.4328
2026-02-24 12:29:21.699 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=60 epoch=0 loss=12.2541
2026-02-24 12:29:22.553 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=70 epoch=0 loss=12.7307
2026-02-24 12:29:23.416 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=80 epoch=0 loss=13.0257
2026-02-24 12:29:24.271 | INFO     | neuralrec.run.callbacks.logging:on_step_end

In [ ]:
from neuralrec.data.dataloader import PrefetchDataLoader


train_loader = PrefetchDataLoader(
    dataset=train_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=8,  # increase number of workers
    pin_memory=True,  # transfer tensors in pinned memory
    collate_fn=pad_collate_item_id,
    device=torch.device("cuda"),  # move tensors to GPU
    buffer_size=4,  # prefetch 4 batches
)

model = RecommenderTransformer(
    num_items=max_item_id,
    d_model=config.d_model,
    encoder=FlashTransformerEncoder(
        d_model=config.d_model,
        nhead=config.n_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
        causal=True,
    )
).cuda()
model = AutoCast(model)  # if using fp16, use GradScaler

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, fused=True)

TrainRunner(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    callbacks=[
        GradientNormClippingCallback(config.gradient_clipping),
        LoggingCallback().every_n_steps(config.logging_every_n_steps),
    ],
).fit(num_epochs=config.num_epochs)

2026-02-24 12:31:18.824 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=10 epoch=0 loss=13.4475
2026-02-24 12:31:19.641 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=20 epoch=0 loss=13.5709
2026-02-24 12:31:20.458 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=30 epoch=0 loss=13.2050
2026-02-24 12:31:21.272 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=40 epoch=0 loss=12.3948
2026-02-24 12:31:22.085 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=50 epoch=0 loss=11.9986
2026-02-24 12:31:22.900 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=60 epoch=0 loss=11.9042
2026-02-24 12:31:23.717 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=70 epoch=0 loss=12.1935
2026-02-24 12:31:24.535 | INFO     | neuralrec.run.callbacks.logging:on_step_end:33 - step=80 epoch=0 loss=12.5968
2026-02-24 12:31:25.350 | INFO     | neuralrec.run.callbacks.logging:on_step_end

# What is missing
1. Snapshotting (saving checkpoints)
2. Metric reduction in distributed training
3. Gradient Checkpointing
4. Ensure parameters have an optimizer
5. Ensure parameters have gradients
6. Ensure parameters don't have duplicated optimizer groups
7. Gradient accumulation
8. ...